# 03 ADS-B fetch

Downloads the ADSB.lol daily archives (ODbL) and keeps only the positions near the ten airports in a window from 6 h
before to 30 min after each departure's take-off. The result is one filtered Parquet file per month in
`data/04_additional_data/adsb_lol`.

A full run takes several hours and was run as a Slurm batch job with
`jupyter nbconvert --to notebook --execute`. Days that are already finished are skipped, so an interrupted run
continues where it stopped. A short date range is enough to try it on a single day.

In [1]:
# CONFIGURATION — inclusive UTC movement dates
START_DATE = "2025-01-01"
END_DATE = "2026-07-31"

# Completed archive days are skipped unless explicitly forced. Failed/incomplete days retry.
FORCE_REPROCESS = False
FORCE_REPROCESS_DATES = []  # e.g. ["2025-01-02", "2025-01-03"] (source/archive dates)
# Forces apply only to archive dates required by this run, including midnight boundary days.
# Replacement is atomic and removes the old source_date rows, never appending duplicates.

# Number of archive days processed in parallel (1 = one after the other, at most 8).
# Effective count is capped by CPU affinity, SLURM_CPUS_PER_TASK, and a hard limit of 8.
MAX_WORKERS = 8

BATCH_ROWS = 20_000
MAX_TRACE_BYTES = 512 * 1024 * 1024  # decompressed per-aircraft limit; fail visibly if exceeded
HTTP_TIMEOUT = (30, 120)
PROGRESS_INTERVAL_SECONDS = 30  # progress updates during download/filtering

# False during normal fetching.
# True runs the slow full integrity check of all saved data.
RUN_FULL_VALIDATION = False

# A GITHUB_TOKEN environment variable, if present, raises the GitHub API limits.

In [2]:
import concurrent.futures as futures
import importlib
import multiprocessing as mp
import queue
import shutil
import sys
import collections
import contextlib
import datetime as dt
import fcntl
import gzip
import hashlib
import io
import json
import math
import os
from pathlib import Path
import re
import sqlite3
import tarfile
import tempfile
import time
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from IPython.display import display
from traffic.data import airports

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/02_clean/train_DEP_merged_sorted.parquet").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside the PRC_Data_Challenge_2026 repository")
OUT = ROOT / "data/04_additional_data/adsb_lol"
OUT.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = OUT / "download_progress.json"
# 2025 training departures and the 2026 final ranking departures (a superset of ranking.parquet).
INPUTS = [ROOT / "data/02_clean/train_DEP_merged_sorted.parquet",
          ROOT / "data/02_clean/final_ranking.parquet"]
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "PRC-2026-filtered-ADSB-acquisition"})
UTC = dt.timezone.utc
def now():
    return dt.datetime.now(UTC).isoformat()

def require(condition, message):
    if not condition:
        raise ValueError(message)

def norm(value):
    if value is None or pd.isna(value):
        return ""
    value = str(value).strip().upper()
    return "" if value in {"", "NAN", "NONE", "UNKNOWN", "N/A"} else value

def identifier(value):
    if pd.isna(value):
        return None
    return str(int(value)) if isinstance(value, (float, int)) else str(value)

start = pd.Timestamp(START_DATE, tz="UTC")
end = pd.Timestamp(END_DATE, tz="UTC")
require(start <= end and start == start.normalize() and end == end.normalize(),
        "START_DATE and END_DATE must be ordered calendar dates")
force_dates = {dt.date.fromisoformat(d).isoformat() for d in FORCE_REPROCESS_DATES}


def worker_capacity(requested):
    require(isinstance(requested, int) and not isinstance(requested, bool) and requested >= 1,
            "MAX_WORKERS must be a positive integer")
    try:
        affinity_cpus = len(os.sched_getaffinity(0))
    except (AttributeError, OSError):
        affinity_cpus = os.cpu_count() or 1
    slurm_text = os.environ.get("SLURM_CPUS_PER_TASK")
    slurm_cpus = int(slurm_text) if slurm_text else None
    require(slurm_cpus is None or slurm_cpus >= 1, "Invalid SLURM_CPUS_PER_TASK")
    available = min(affinity_cpus, slurm_cpus) if slurm_cpus is not None else affinity_cpus
    return min(requested, available, 8), affinity_cpus, slurm_cpus

WORKER_LIMIT, AFFINITY_CPUS, SLURM_CPUS = worker_capacity(MAX_WORKERS)


## Departures and archive days

Matching uses the exact callsign (or the flight number when the callsign is missing), the airport area and the
window from 6 h before to 30 min after take-off. Ambiguous matches are dropped rather than guessed.

In [3]:
required = ["MVT_ID_mvt", "FLIGHT_ID_mvt", "ADEP_mvt", "MVT_TIME_UTC_mvt",
            "PHASE_mvt", "CALLSIGN_flt", "FLIGHT_mvt"]
frames = []
for path, year in zip(INPUTS, [2025, 2026]):
    names = pq.read_schema(path).names
    print(path.relative_to(ROOT), "identifier columns:",
          [c for c in names if any(k in c.upper() for k in
                                  ["ID_", "CALLSIGN", "REGISTRATION", "ICAO24", "HEX", "FLIGHT_MVT"])])
    require(set(required) <= set(names), f"Unexpected schema: {path}")
    frame = pd.read_parquet(path, columns=required)
    frame = frame.loc[frame.PHASE_mvt.eq("DEP")].copy()
    frame["MVT_TIME_UTC_mvt"] = pd.to_datetime(frame.MVT_TIME_UTC_mvt, utc=True, errors="raise")
    require(frame.MVT_TIME_UTC_mvt.notna().all(), "Missing movement timestamp")
    frame = frame.loc[frame.MVT_TIME_UTC_mvt.dt.year.eq(year)]
    frames.append(frame)
all_flights = pd.concat(frames, ignore_index=True)
for name in ["MVT_ID_mvt", "FLIGHT_ID_mvt"]:
    all_flights[name] = all_flights[name].map(identifier)
require(all_flights.MVT_ID_mvt.notna().all() and not all_flights.MVT_ID_mvt.duplicated().any(),
        "Movement identifiers must be non-null and globally unique")
all_flights["match_callsign"] = all_flights.CALLSIGN_flt.map(norm)
missing = all_flights.match_callsign.eq("")
all_flights.loc[missing, "match_callsign"] = all_flights.loc[missing, "FLIGHT_mvt"].map(norm)
all_flights["window_start"] = all_flights.MVT_TIME_UTC_mvt - pd.Timedelta(hours=6)
all_flights["window_end"] = all_flights.MVT_TIME_UTC_mvt + pd.Timedelta(minutes=30)
selected = all_flights.loc[all_flights.MVT_TIME_UTC_mvt.between(start, end + pd.Timedelta(days=1),
                                                                          inclusive="left")].copy()
required_days = set()
for left, right in zip(selected.window_start, selected.window_end):
    required_days.update(d.date().isoformat() for d in pd.date_range(left.normalize(), right.normalize()))
required_days = sorted(required_days)
print(f"Selected UTC range: {START_DATE} → {END_DATE}; calendar days: {(end-start).days+1}")
print("Relevant challenge departures:", len(selected))
print("Airports:", sorted(selected.ADEP_mvt.dropna().unique()))
print("Years/months:", sorted(selected.MVT_TIME_UTC_mvt.dt.strftime("%Y-%m").unique()))
print("Departures with no usable callsign:", int(selected.match_callsign.eq("").sum()))
# Metadata only: this does not load any trajectory archive.
if required_days:
    lo = pd.Timestamp(required_days[0], tz="UTC")
    hi = pd.Timestamp(required_days[-1], tz="UTC") + pd.Timedelta(days=1)
    relevant = all_flights.loc[(all_flights.window_start < hi) & (all_flights.window_end >= lo)].copy()
else:
    relevant = all_flights.iloc[:0].copy()


data/02_clean/train_DEP_merged_sorted.parquet identifier columns: ['MVT_ID_mvt', 'FLIGHT_ID_mvt', 'FLIGHT_mvt', 'CALLSIGN_flt']


data/02_clean/final_ranking.parquet identifier columns: ['MVT_ID_mvt', 'FLIGHT_ID_mvt', 'FLIGHT_mvt', 'CALLSIGN_flt']


Selected UTC range: 2025-01-01 → 2026-07-31; calendar days: 577
Relevant challenge departures: 2755837
Airports: ['EDDF', 'EDDM', 'EGLL', 'EHAM', 'LEBL', 'LEMD', 'LFPG', 'LIRF', 'LSZH', 'LTFM']


Years/months: ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-06', '2026-07']
Departures with no usable callsign: 106


In [4]:
# Airport geometry is computed once per airport and reused for every source day.
bounds_cache = {}
for airport_str in sorted(relevant.ADEP_mvt.dropna().unique()):
    try:
        airport = airports[airport_str]
        shape = airport.shape if airport is not None else None
        require(shape is not None and not shape.is_empty and shape.is_valid, "missing/invalid geometry")
        bounds = shape.convex_hull.buffer(0.1)
        min_lon, min_lat, max_lon, max_lat = bounds.bounds
        require(np.isfinite(bounds.bounds).all() and -180 <= min_lon < max_lon <= 180
                and -90 <= min_lat < max_lat <= 90, "invalid geographic bounds")
        bounds_cache[airport_str] = bounds.bounds
    except Exception as exc:
        raise RuntimeError(f"No valid traffic airport geometry for {airport_str}: {exc}") from exc
require(relevant.ADEP_mvt.notna().all(), "Challenge departure has no airport")
display(pd.DataFrame([{"ADEP_mvt": a, **dict(zip(
    ["min_lon", "min_lat", "max_lon", "max_lat"], b))} for a, b in bounds_cache.items()]))

def day_flights(day):
    left = pd.Timestamp(day, tz="UTC")
    right = left + pd.Timedelta(days=1)
    result = all_flights.loc[(all_flights.window_start < right) & (all_flights.window_end >= left)].copy()
    for i, name in enumerate(["min_lon", "min_lat", "max_lon", "max_lat"]):
        result[name] = result.ADEP_mvt.map(lambda a: bounds_cache[a][i])
    return result

def read_progress():
    if not PROGRESS_PATH.exists():
        return {"version": 1, "days": {}}
    result = json.loads(PROGRESS_PATH.read_text())
    require(result.get("version") == 1 and isinstance(result.get("days"), dict), "Invalid progress file")
    return result

def execution_plan(progress):
    done, todo, forced, retry = [], [], [], []
    for day in required_days:
        old = progress["days"].get(day, {})
        force = FORCE_REPROCESS or day in force_dates
        # A finished day is redone when new departures now fall into it (e.g. a newly released month).
        grown = old.get("challenge_departures", 0) < len(day_flights(day))
        if old.get("status") == "completed" and not force and not grown:
            require((OUT / f"{day[:7]}.parquet").exists(),
                    f"{day}: completed output is missing; restore it or explicitly force this date")
            done.append(day)
        else:
            todo.append(day)
            if force or (old.get("status") == "completed" and grown):
                forced.append(day)
            if old.get("status") in {"failed", "processing"}:
                retry.append(day)
    return done, todo, forced, retry

progress = read_progress()
skipped, to_process, forced, retry = execution_plan(progress)
print(f"Configured range: {START_DATE} → {END_DATE}")
print(f"Required archive dates: {len(required_days)} (includes UTC boundary days)")
print(f"Already completed: {len(skipped)}; To process: {len(to_process)}; "
      f"Forced reprocessing: {len(forced)}; Failed/incomplete to retry: {len(retry)}")
print(f"Workers: requested={MAX_WORKERS}, CPU affinity={AFFINITY_CPUS}, "
      f"SLURM_CPUS_PER_TASK={SLURM_CPUS}, active at most={min(WORKER_LIMIT, len(to_process))}")
print("Dates to process:\n" + "\n".join(to_process))
print("Force dates outside this plan (ignored):", sorted(force_dates - set(required_days)))
if to_process:
    display(day_flights(to_process[0]).head())


Invalid geometry with id 3582389: no outer element


Invalid geometry with id 3766492: no outer element


Invalid geometry with id 8473008: no outer element


,ADEP_mvt,min_lon,min_lat,max_lon,max_lat
0,EDDF,8.396802,49.897950,8.701181,50.152189
1,EDDM,11.645316,48.240366,11.925790,48.467182
2,EGLL,-0.590842,51.356796,-0.315966,51.577679
3,EHAM,4.608811,52.188063,4.909087,52.462655
4,LEBL,1.955767,41.180208,2.209693,41.413461
5,LEMD,-3.697411,40.347817,-3.426470,40.642215
6,LFPG,2.392272,48.883993,2.732856,49.130009
7,LIRF,12.126262,41.681483,12.375743,41.946379
8,LSZH,8.434742,47.340614,8.676169,47.583046
9,LTFM,28.600131,41.148369,28.904050,41.402137


Configured range: 2025-01-01 → 2026-07-31
Required archive dates: 489 (includes UTC boundary days)
Already completed: 489; To process: 0; Forced reprocessing: 0; Failed/incomplete to retry: 0
Workers: requested=8, CPU affinity=8, SLURM_CPUS_PER_TASK=8, active at most=0
Dates to process:

Force dates outside this plan (ignored): []


## Download

Each worker streams one archive day, keeps the matching positions and hands them to the main process, which merges
them into the month file. A day is marked finished only after its data are written.

In [5]:
# Worker functions live in an importable module so spawn works from a Jupyter kernel.
# Child processes import only this module; they do not reload the challenge datasets.
worker_directory = str(ROOT / "data")
if worker_directory not in sys.path:
    sys.path.insert(0, worker_directory)
import adsb_lol_worker
adsb_lol_worker = importlib.reload(adsb_lol_worker)
SCHEMA = adsb_lol_worker.SCHEMA


In [6]:
def sync_replace(temp, destination):
    with open(temp, "rb") as handle:
        os.fsync(handle.fileno())
    os.replace(temp, destination)
    descriptor = os.open(destination.parent, os.O_RDONLY | os.O_DIRECTORY)
    try:
        os.fsync(descriptor)
    finally:
        os.close(descriptor)

def save_progress(progress):
    temp = OUT / ".adsb-progress.tmp"
    temp.write_text(json.dumps(progress, indent=2, allow_nan=False) + "\n")
    sync_replace(temp, PROGRESS_PATH)

def replace_day(day, daily_path, expected_rows):
    destination = OUT / f"{day[:7]}.parquet"
    print(f"{day} [5/6 Save] {destination.name} is rewritten atomically …", flush=True)
    temp = OUT / ".adsb-month.tmp"
    try:
        with pq.ParquetWriter(temp, SCHEMA, compression="zstd") as writer:
            if destination.exists():
                with pq.ParquetFile(destination) as old:
                    require(old.schema_arrow.equals(SCHEMA), "Existing monthly schema differs")
                    for batch in old.iter_batches(batch_size=BATCH_ROWS):
                        table = pa.Table.from_batches([batch])
                        # Old data survive until the entire replacement has been written and verified.
                        table = table.filter(pa.compute.not_equal(table["source_date"], day))
                        writer.write_table(table)
            rows_written = 0
            with pq.ParquetFile(daily_path) as daily:
                require(daily.schema_arrow.equals(SCHEMA), "Worker daily schema differs")
                for batch in daily.iter_batches(batch_size=BATCH_ROWS):
                    table = pa.Table.from_batches([batch])
                    require(pa.compute.all(pa.compute.equal(table["source_date"], day)).as_py(),
                            "Worker file contains another source day")
                    rows_written += table.num_rows
                    writer.write_table(table)
            require(rows_written == expected_rows, "Worker daily row count mismatch")
        print(f"{day} [6/6 Check] Reading and checking the new monthly file …", flush=True)
        # Fully read each replacement batch before making it visible.
        with pq.ParquetFile(temp) as check:
            require(check.schema_arrow.equals(SCHEMA), "Replacement schema mismatch")
            for batch in check.iter_batches(batch_size=BATCH_ROWS):
                batch.validate(full=True)
        sync_replace(temp, destination)
        print(f"{day} [6/6 Saved] {destination.name} replaced safely.", flush=True)
    finally:
        temp.unlink(missing_ok=True)


In [7]:
def drain_worker_logs(events):
    # Limit each pass so continuous log messages cannot starve result handling.
    for _ in range(512):
        try:
            print(events.get_nowait(), flush=True)
        except queue.Empty:
            break

def run_downloads():
    successes, failures = [], []
    with open(OUT / ".download.lock", "a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        for stale in OUT.glob(".adsb-day-*"):
            if stale.is_dir():
                shutil.rmtree(stale)
        for name in [".adsb-month.tmp", ".adsb-progress.tmp"]:
            (OUT / name).unlink(missing_ok=True)
        progress = read_progress()
        skipped, to_process, forced, retry = execution_plan(progress)
        for day in skipped:
            print(f"{day}: already completed, skipped", flush=True)
        capacity, _, _ = worker_capacity(MAX_WORKERS)
        active_workers = min(capacity, len(to_process))
        print(f"Download workers: {active_workers}; open archive days: {len(to_process)}", flush=True)
        if not to_process:
            return progress, skipped, successes, failures, forced

        config = {"batch_rows": BATCH_ROWS, "max_trace_bytes": MAX_TRACE_BYTES,
                  "http_timeout": HTTP_TIMEOUT, "progress_interval_seconds": PROGRESS_INTERVAL_SECONDS}
        context = mp.get_context("spawn")
        events = context.Queue(maxsize=512)
        stop = context.Event()
        pending = {}
        iterator = iter(enumerate(to_process, 1))
        with tempfile.TemporaryDirectory(prefix=".adsb-day-run-", dir=OUT) as run_dir:
            executor = futures.ProcessPoolExecutor(max_workers=active_workers, mp_context=context,
                initializer=adsb_lol_worker.initialize_worker,
                initargs=(config, events, stop, os.getpid()))
            try:
                def submit_next():
                    item = next(iterator, None)
                    if item is None:
                        return False
                    number, day = item
                    flights = day_flights(day)
                    temp_dir = Path(run_dir) / day
                    temp_dir.mkdir()
                    record = {"status": "processing", "successfully_completed": False, "release": None,
                              "processing_timestamp": now(), "challenge_departures": len(flights),
                              "candidate_aircraft": 0, "matched_departures": 0, "retained_observations": 0,
                              "error": None}
                    progress["days"][day] = record
                    save_progress(progress)
                    print(f"=== Day {number}/{len(to_process)}: {day} started ===", flush=True)
                    job = executor.submit(adsb_lol_worker.process_day, day, flights, str(temp_dir))
                    pending[job] = (day, time.monotonic(), temp_dir)
                    return True

                for _ in range(active_workers):
                    submit_next()
                while pending:
                    finished, _ = futures.wait(pending, timeout=0.5, return_when=futures.FIRST_COMPLETED)
                    drain_worker_logs(events)
                    for job in sorted(finished, key=lambda j: pending[j][0]):
                        day, started, temp_dir = pending[job]
                        record = progress["days"][day]
                        try:
                            result = job.result()
                            daily_path = Path(result.pop("day_file"))
                            require(daily_path.resolve() == (temp_dir / "filtered.parquet").resolve(),
                                    "Worker returned an unexpected staging path")
                            record.update(result)
                            # This is the only monthly writer; other workers only stage their own days.
                            replace_day(day, daily_path, record["retained_observations"])
                        except Exception as exc:
                            record.update(status="failed", successfully_completed=False,
                                          error=f"{type(exc).__name__}: {exc}")
                            failures.append(day)
                        else:
                            record.update(status="completed", successfully_completed=True, error=None)
                            successes.append(day)
                        record["processing_timestamp"] = now()
                        record["processing_seconds"] = round(time.monotonic() - started, 1)
                        save_progress(progress)
                        print(f"{day} [Done] " + str({k: record[k] for k in
                              ["status", "release", "challenge_departures", "candidate_aircraft",
                               "matched_departures", "retained_observations", "processing_seconds", "error"]}), flush=True)
                        shutil.rmtree(temp_dir)
                        del pending[job]
                    # Never queue the entire date range: at most active_workers private days exist.
                    while len(pending) < active_workers and submit_next():
                        pass
            except BaseException as exc:
                stop.set()
                for job in pending:
                    job.cancel()
                for day in to_process:
                    record = progress["days"].get(day, {})
                    if record.get("status") == "processing":
                        record.update(error=f"Interrupted: {type(exc).__name__}: {exc}",
                                      processing_timestamp=now())
                save_progress(progress)
                print("Run interrupted; stopping workers. Unfinished days can be repeated.", flush=True)
                raise
            finally:
                # Keep the run lock and staging directory until all workers have stopped.
                executor.shutdown(wait=True, cancel_futures=True)
                drain_worker_logs(events)
                events.close()
                events.join_thread()
    return progress, skipped, successes, failures, forced

def print_run_summary(progress, skipped, successes, failures, forced):
    selected_ids = set(selected.MVT_ID_mvt)
    def ids_for(days):
        return set().union(*(set(progress["days"][d].get("matched_movement_ids", [])) for d in days))
    completed_days = [d for d, r in progress["days"].items() if r["status"] == "completed"]
    cumulative_ids = ids_for(completed_days)
    processed_ids = ids_for(successes)
    print("Current run:", {"requested_archive_dates": len(required_days), "skipped": len(skipped),
          "successful": len(successes), "forced_successful": len(set(successes) & set(forced)),
          "failed": len(failures), "selected_challenge_departures": len(selected_ids),
          "selected_matches_in_newly_processed_days": len(processed_ids & selected_ids),
          "all_matches_in_newly_processed_days_including_boundary": len(processed_ids),
          "retained_observations_in_processed_days": sum(progress["days"][d]["retained_observations"] for d in successes)})
    selected_matches = len(cumulative_ids & selected_ids)
    require(selected_matches <= len(selected_ids), "Selected coverage exceeds its departure denominator")
    print(f"Selected movement period {START_DATE} → {END_DATE} (all completed days, incl. skipped): "
          f"{selected_matches:,} / {len(selected_ids):,} departures matched")
    print("Cumulative progress — all archived movements, including dates outside the selection:",
          {"statuses": dict(collections.Counter(r["status"] for r in progress["days"].values())),
           "challenge_departure_day_pairs": sum(progress["days"][d]["challenge_departures"] for d in completed_days),
           "unique_matched_departures_all_movement_dates": len(cumulative_ids),
           "matched_departures_outside_selected_range": len(cumulative_ids - selected_ids),
           "retained_observations_all_movement_dates": sum(progress["days"][d]["retained_observations"] for d in completed_days)})
    if failures:
        print("Failed dates remain retryable:", failures)

run_started = now()
progress, skipped, successes, failures, forced = run_downloads()
print_run_summary(progress, skipped, successes, failures, forced)


2024-12-31: already completed, skipped


2025-01-01: already completed, skipped


2025-01-02: already completed, skipped


2025-01-03: already completed, skipped


2025-01-04: already completed, skipped


...

2026-07-31: already completed, skipped


2026-08-01: already completed, skipped


Download workers: 0; open archive days: 0


Current run: {'requested_archive_dates': 489, 'skipped': 489, 'successful': 0, 'forced_successful': 0, 'failed': 0, 'selected_challenge_departures': 2755837, 'selected_matches_in_newly_processed_days': 0, 'all_matches_in_newly_processed_days_including_boundary': 0, 'retained_observations_in_processed_days': 0}
Selected movement period 2025-01-01 → 2026-07-31 (all completed days, incl. skipped): 2,346,159 / 2,755,837 departures matched


Cumulative progress — all archived movements, including dates outside the selection: {'statuses': {'completed': 489}, 'challenge_departure_day_pairs': 3002226, 'unique_matched_departures_all_movement_dates': 2346159, 'matched_departures_outside_selected_range': 0, 'retained_observations_all_movement_dates': 308023209}


In [8]:
print(f"Active limit: {MAX_TRACE_BYTES / 1024**2:.0f} MiB")

Active limit: 512 MiB


## Check the saved data and coverage

In [9]:
def validate_outputs():
    flight_index = all_flights.set_index("MVT_ID_mvt")
    counts = collections.Counter()
    examples = []
    source_counts = collections.Counter()

    with tempfile.TemporaryDirectory(prefix=".adsb-day-validation-", dir=OUT) as temp_dir:
        db = sqlite3.connect(Path(temp_dir) / "keys.sqlite")

        try:
            db.execute("PRAGMA cache_size=-8192")

            db.execute(
                "CREATE TABLE keys (mvt TEXT, hex TEXT, ts TEXT, lat REAL, lon REAL, "
                "PRIMARY KEY(mvt,hex,ts,lat,lon)) WITHOUT ROWID"
            )

            for path in sorted(OUT.glob("????-??.parquet")):
                with pq.ParquetFile(path) as parquet:
                    require(
                        parquet.schema_arrow.equals(SCHEMA),
                        f"Schema mismatch: {path}"
                    )

                    for batch in parquet.iter_batches(batch_size=BATCH_ROWS):
                        frame = batch.to_pandas()

                        require(
                            str(frame.timestamp.dt.tz) == "UTC"
                            and str(frame.MVT_TIME_UTC_mvt.dt.tz) == "UTC",
                            "Non-UTC timestamps"
                        )

                        require(
                            frame.timestamp.notna().all(),
                            "Missing timestamp"
                        )

                        require(
                            frame.MVT_ID_mvt.isin(flight_index.index).all(),
                            "Unassociated observations"
                        )

                        require(
                            frame.source_date.eq(
                                frame.timestamp.dt.strftime("%Y-%m-%d")
                            ).all(),
                            "Observation outside source UTC date"
                        )

                        require(
                            frame.source_date.str[:7].eq(path.stem).all(),
                            "Wrong monthly partition"
                        )

                        meta = (
                            flight_index
                            .loc[frame.MVT_ID_mvt]
                            .reset_index(drop=True)
                        )

                        require(
                            frame.ADEP_mvt.eq(meta.ADEP_mvt).all()
                            and frame.MVT_TIME_UTC_mvt.eq(meta.MVT_TIME_UTC_mvt).all(),
                            "Flight metadata mismatch"
                        )

                        require(
                            frame.callsign.eq(meta.match_callsign).all(),
                            "Callsign association mismatch"
                        )

                        require(
                            (frame.timestamp >= meta.window_start).all()
                            and (frame.timestamp <= meta.window_end).all(),
                            "Outside flight window"
                        )

                        for airport_str in frame.ADEP_mvt.unique():
                            if airport_str not in bounds_cache:
                                shape = airports[airport_str].shape

                                require(
                                    shape is not None
                                    and not shape.is_empty
                                    and shape.is_valid,
                                    f"Missing/invalid geometry for {airport_str}"
                                )

                                bounds_cache[airport_str] = (
                                    shape.convex_hull.buffer(0.1).bounds
                                )

                            west, south, east, north = bounds_cache[airport_str]

                            part = frame.loc[
                                frame.ADEP_mvt.eq(airport_str)
                            ]

                            require(
                                np.isfinite(
                                    part[["latitude", "longitude"]]
                                ).all().all()
                                and part.longitude.between(west, east).all()
                                and part.latitude.between(south, north).all(),
                                f"Outside bounds: {airport_str}"
                            )

                        keys = [
                            (
                                r.MVT_ID_mvt,
                                r.icao24,
                                r.timestamp.isoformat(),
                                r.latitude,
                                r.longitude,
                            )
                            for r in frame.itertuples(index=False)
                        ]

                        # Duplicates raise sqlite3.IntegrityError.
                        db.executemany(
                            "INSERT INTO keys VALUES (?,?,?,?,?)",
                            keys
                        )

                        counts.update(
                            frame.MVT_ID_mvt.value_counts().to_dict()
                        )

                        source_counts.update(
                            frame.source_date.value_counts().to_dict()
                        )

                        if sum(len(x) for x in examples) < 10:
                            examples.append(
                                frame.loc[
                                    frame.MVT_ID_mvt.isin(
                                        selected.MVT_ID_mvt
                                    )
                                ].head(10)
                            )

                db.commit()
                print(f"Validated {path.name}")

            # A flight must not acquire conflicting aircraft across midnight either.
            conflicts = db.execute(
                "SELECT mvt FROM keys "
                "GROUP BY mvt "
                "HAVING COUNT(DISTINCT hex)>1 "
                "LIMIT 10"
            ).fetchall()

            require(
                not conflicts,
                "Cross-day aircraft identity conflicts; "
                f"inspect and force affected dates: {conflicts}"
            )

        finally:
            db.close()

    for day, record in read_progress()["days"].items():
        if record["status"] == "completed":
            require(
                source_counts[day] == record["retained_observations"],
                f"Progress/output count mismatch: {day}"
            )

    return counts, examples


if RUN_FULL_VALIDATION:
    print("Running full persisted-data validation ...")

    # Do not validate while another session is replacing files.
    with open(OUT / ".download.lock", "a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        observation_counts, examples = validate_outputs()

    coverage = selected[
        ["MVT_ID_mvt", "ADEP_mvt", "MVT_TIME_UTC_mvt"]
    ].copy()

    coverage["observations"] = (
        coverage.MVT_ID_mvt
        .map(observation_counts)
        .fillna(0)
        .astype(int)
    )

    coverage["year_month"] = (
        coverage.MVT_TIME_UTC_mvt.dt.strftime("%Y-%m")
    )

    def coverage_table(key):
        records = []

        for value, group in coverage.groupby(key):
            n = group.observations

            records.append({
                key: value,
                "challenge_departures": len(n),
                "matched": int(n.gt(0).sum()),
                "unmatched": int(n.eq(0).sum()),
                "match_pct": 100 * n.gt(0).mean(),
                "median_points_matched": n[n.gt(0)].median(),
                **{
                    f"pct_at_least_{k}": 100 * n.ge(k).mean()
                    for k in [10, 30, 60]
                },
            })

        return pd.DataFrame(records)

    selected_matched = int(
        coverage.observations.gt(0).sum()
    )

    require(
        selected_matched <= len(coverage),
        "Selected coverage exceeds departure count"
    )

    print(
        f"Validated selection {START_DATE} → {END_DATE}: "
        f"{selected_matched:,} / {len(coverage):,} departures matched"
    )

    print(
        "Coverage tables: only departures within "
        "the selected movement period."
    )

    display(coverage_table("ADEP_mvt"))
    display(coverage_table("year_month"))

    print(
        "Required archive statuses:",
        dict(
            collections.Counter(
                read_progress()["days"]
                .get(d, {})
                .get("status", "not_processed")
                for d in required_days
            )
        )
    )

    if examples:
        display(
            pd.concat(
                examples,
                ignore_index=True
            ).head(10)
        )
    else:
        print("No retained observations to display.")


else:
    print(
        "RUN_FULL_VALIDATION=False — "
        "full Parquet/SQLite validation skipped."
    )

    progress_now = read_progress()
    selected_ids = set(selected.MVT_ID_mvt)

    completed_required_days = [
        day
        for day in required_days
        if progress_now["days"]
        .get(day, {})
        .get("status") == "completed"
    ]

    matched_ids = set()

    for day in completed_required_days:
        matched_ids.update(
            progress_now["days"][day].get(
                "matched_movement_ids",
                []
            )
        )

    matched_selected_ids = matched_ids & selected_ids

    coverage = selected[
        ["MVT_ID_mvt", "ADEP_mvt", "MVT_TIME_UTC_mvt"]
    ].copy()

    coverage["matched"] = (
        coverage.MVT_ID_mvt.isin(
            matched_selected_ids
        )
    )

    coverage["year_month"] = (
        coverage.MVT_TIME_UTC_mvt.dt.strftime("%Y-%m")
    )

    def coverage_table(key):
        records = []

        for value, group in coverage.groupby(key):
            total = len(group)
            matched = int(group.matched.sum())

            records.append({
                key: value,
                "challenge_departures": total,
                "matched": matched,
                "unmatched": total - matched,
                "match_pct": (
                    100 * matched / total
                    if total
                    else np.nan
                ),
            })

        return pd.DataFrame(records)

    print(
        f"Selection {START_DATE} → {END_DATE}: "
        f"{len(matched_selected_ids):,} / "
        f"{len(selected_ids):,} departures matched according to progress"
    )

    print(
        "Coverage from download_progress.json; "
        "no complete Parquet files read."
    )

    display(coverage_table("ADEP_mvt"))
    display(coverage_table("year_month"))

    print(
        "Required archive statuses:",
        dict(
            collections.Counter(
                progress_now["days"]
                .get(d, {})
                .get("status", "not_processed")
                for d in required_days
            )
        )
    )

RUN_FULL_VALIDATION=False — full Parquet/SQLite validation skipped.


Selection 2025-01-01 → 2026-07-31: 2,346,159 / 2,755,837 departures matched according to progress
Coverage from download_progress.json; no complete Parquet files read.


,ADEP_mvt,challenge_departures,matched,unmatched,match_pct
0,EDDF,300812,292530,8282,97.246785
1,EDDM,218206,212326,5880,97.305299
2,EGLL,316767,311270,5497,98.264655
3,EHAM,323816,314362,9454,97.080441
4,LEBL,239019,234198,4821,97.983006
5,LEMD,284722,261538,23184,91.857320
6,LFPG,317072,308487,8585,97.292413
7,LIRF,212850,181517,31333,85.279305
8,LSZH,179815,175085,4730,97.369519
9,LTFM,362758,54846,307912,15.119170


,year_month,challenge_departures,matched,unmatched,match_pct
0,2025-01,153706,127553,26153,82.985049
1,2025-02,143732,115921,27811,80.650795
2,2025-03,164449,144026,20423,87.580952
3,2025-04,175288,142237,33051,81.144745
4,2025-05,185202,151000,34202,81.532597
5,2025-06,183142,158774,24368,86.694478
6,2025-07,190713,163014,27699,85.476082
7,2025-08,191182,163247,27935,85.388269
8,2025-09,183950,157225,26725,85.471596
9,2025-10,185674,161626,24048,87.048267


Required archive statuses: {'completed': 489}


## Source

ADSB.lol `globe_history` (https://github.com/adsblol), Open Database License 1.0, which asks for the ADSB.lol
attribution when derived data are shared. Releases, repositories and access times per day are stored in
`download_progress.json`.

In [10]:
provenance = read_progress()
print("Selected movement range:", START_DATE, "→", END_DATE)
print("This run started (UTC):", run_started)
print("Source years fetched:", sorted({d[:4] for d, r in provenance["days"].items() if r["status"] == "completed"}))
print("Repositories used:", sorted({r["repository"] for r in provenance["days"].values() if r.get("repository")}))
print("Access timestamps (UTC):", sorted({r["processing_timestamp"] for r in provenance["days"].values()}))


Selected movement range: 2025-01-01 → 2026-07-31
This run started (UTC): 2026-10-09T20:35:49.077620+00:00
Source years fetched: ['2024', '2025', '2026']
Repositories used: ['adsblol/globe_history_2025', 'adsblol/globe_history_2026']
Access timestamps (UTC): ['2026-09-27T18:31:16.606225+00:00', '2026-09-27T18:31:19.517172+00:00', '2026-09-27T18:31:25.762251+00:00', '2026-09-27T18:31:32.824011+00:00', '2026-09-27T18:31:34.610968+00:00', '2026-09-27T18:31:50.312918+00:00', '2026-09-27T18:32:05.056318+00:00', '2026-09-27T18:32:08.750132+00:00', '2026-09-27T18:33:59.111491+00:00', '2026-09-27T18:34:03.574179+00:00', '2026-09-27T18:34:08.167056+00:00', '2026-09-27T18:34:44.966986+00:00', '2026-09-27T18:35:19.679548+00:00', '2026-09-27T18:35:29.911418+00:00', '2026-09-27T18:35:58.357737+00:00', '2026-09-27T18:36:53.176189+00:00', '2026-09-27T18:37:45.366250+00:00', '2026-09-27T18:37:56.903036+00:00', '2026-09-27T18:38:33.135408+00:00', '2026-09-27T18:38:41.804226+00:00', '2026-09-27T18:38:56.